# Notebook 05 — Кластеризація об'єктів за доступністю (ML: Unsupervised)

Використовуємо методи машинного навчання без вчителя для виявлення природних груп об'єктів.

## Методи
- **KMeans** (k=4) — чіткі, рівновіддалені кластери
- **DBSCAN** — кластери довільної форми + виявлення викидів

## Метрики оцінювання
- **Silhouette Score** — наскільки об'єкти схожі всередині кластеру vs між кластерами (−1..+1, вище = краще)
- **Davies-Bouldin Index** — відношення розкиду всередині кластеру до відстані між кластерами (нижче = краще)

In [13]:
from config_loader import cfg, KMEANS_K, RANDOM_STATE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('Бібліотеки завантажено')

In [14]:
df = pd.read_csv('../data/processed/accessibility_scores.csv', encoding='utf-8')

print(f'Об\'єктів: {len(df)}')
print(f'Колонки: {list(df.columns)}')
df[['score_peak', 'score_offpeak', 'score_delta', 'n_stops', 'n_routes']].describe().round(3)

## 1. Підготовка ознак

Для кластеризації використовуємо нормалізовані компоненти індексу та сам індекс.

In [15]:
FEATURE_COLS_PEAK    = ['n_stops', 'n_routes', 'frequency_peak',    'score_peak']
FEATURE_COLS_OFFPEAK = ['n_stops', 'n_routes', 'frequency_offpeak', 'score_offpeak']

df_clean = df.dropna(subset=FEATURE_COLS_PEAK + ['score_offpeak']).copy()
print(f'Об\'єктів після очищення: {len(df_clean)}')

X_peak    = df_clean[FEATURE_COLS_PEAK].values
X_offpeak = df_clean[FEATURE_COLS_OFFPEAK].values

scaler_peak    = StandardScaler()
scaler_offpeak = StandardScaler()
X_peak_scaled    = scaler_peak.fit_transform(X_peak)
X_offpeak_scaled = scaler_offpeak.fit_transform(X_offpeak)

print(f'Матриця ознак (пік):    {X_peak_scaled.shape}  — {FEATURE_COLS_PEAK}')
print(f'Матриця ознак (міжпік): {X_offpeak_scaled.shape}  — {FEATURE_COLS_OFFPEAK}')

## 2. Визначення оптимального k (метод ліктя + Silhouette)

In [16]:
inertias = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_peak_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_peak_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'bo-', markersize=8)
ax1.set_title('Метод ліктя (пікові дані)', fontsize=13)
ax1.set_xlabel('Кількість кластерів k')
ax1.set_ylabel('Інерція (сума квадратів відстаней)')
ax1.axvline(4, color='red', linestyle='--', alpha=0.7, label='k=4 (обраний)')
ax1.legend()

ax2.plot(k_range, silhouette_scores, 'gs-', markersize=8)
ax2.set_title('Silhouette Score по k (пікові дані)', fontsize=13)
ax2.set_xlabel('Кількість кластерів k')
ax2.set_ylabel('Silhouette Score')
ax2.axvline(4, color='red', linestyle='--', alpha=0.7, label='k=4 (обраний)')
ax2.legend()

plt.tight_layout()
plt.savefig('../outputs/07_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = k_range[np.argmax(silhouette_scores)]
print(f'Оптимальне k за Silhouette: {best_k}')
print(f'Силует при k=4: {silhouette_scores[2]:.3f}')

## 3. KMeans (k=4)

Чотири кластери відповідають чотирьом рівням доступності: відмінна, добра, погана, критична.

In [17]:
K = KMEANS_K

# ── KMeans для ПІКОВОГО індексу ───────────────────────────────────────────────
kmeans_peak = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
df_clean['cluster_peak'] = kmeans_peak.fit_predict(X_peak_scaled)

km_peak_silhouette = silhouette_score(X_peak_scaled, df_clean['cluster_peak'])
km_peak_db         = davies_bouldin_score(X_peak_scaled, df_clean['cluster_peak'])

# ── KMeans для МІЖПІКОВОГО індексу ───────────────────────────────────────────
kmeans_offpeak = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
df_clean['cluster_offpeak'] = kmeans_offpeak.fit_predict(X_offpeak_scaled)

km_offpeak_silhouette = silhouette_score(X_offpeak_scaled, df_clean['cluster_offpeak'])
km_offpeak_db         = davies_bouldin_score(X_offpeak_scaled, df_clean['cluster_offpeak'])

print(f'=== KMeans (k={K}) метрики ===')
print(f'{"":30s} {"Пік":>10s} {"Міжпік":>10s}')
print(f'{"Silhouette Score ↑":30s} {km_peak_silhouette:>10.4f} {km_offpeak_silhouette:>10.4f}')
print(f'{"Davies-Bouldin Index ↓":30s} {km_peak_db:>10.4f} {km_offpeak_db:>10.4f}')

In [18]:
def assign_labels(df_in, cluster_col, score_col):
    """Призначає мітки Критична/Погана/Добра/Відмінна за середнім score."""
    cluster_means = df_in.groupby(cluster_col)[score_col].mean().sort_values()
    names = ['Критична', 'Погана', 'Добра', 'Відмінна']
    label_map = {cid: names[rank] for rank, (cid, _) in enumerate(cluster_means.items())}
    return df_in[cluster_col].map(label_map)

df_clean['label_peak']    = assign_labels(df_clean, 'cluster_peak',    'score_peak')
df_clean['label_offpeak'] = assign_labels(df_clean, 'cluster_offpeak', 'score_offpeak')

# Аліаси для сумісності з NB07
df_clean['cluster_kmeans']      = df_clean['cluster_peak']
df_clean['accessibility_label'] = df_clean['label_peak']

print('=== Розподіл по категоріях (пік) ===')
print(df_clean['label_peak'].value_counts())
print()
print('=== Розподіл по категоріях (міжпік) ===')
print(df_clean['label_offpeak'].value_counts())

## 3б. Аналіз змін кластерів між піком і міжпіком

In [19]:
n_changed = (df_clean['label_peak'] != df_clean['label_offpeak']).sum()
total     = len(df_clean)
print(f'Об\'єктів, що змінили категорію (пік → міжпік): {n_changed} ({100*n_changed/total:.1f}%)')
print()

# Матриця переходів
order = ['Відмінна', 'Добра', 'Погана', 'Критична']
transition = pd.crosstab(
    df_clean['label_peak'], df_clean['label_offpeak'],
    rownames=['Пік'], colnames=['Міжпік']
).reindex(index=order, columns=order, fill_value=0)
print('=== Матриця переходів (рядки = пік, стовпці = міжпік) ===')
print(transition)
print()

# Напрямок змін
label_rank = {'Відмінна': 3, 'Добра': 2, 'Погана': 1, 'Критична': 0}
df_clean['_r_peak']    = df_clean['label_peak'].map(label_rank)
df_clean['_r_offpeak'] = df_clean['label_offpeak'].map(label_rank)
df_clean['_change']    = df_clean['_r_peak'] - df_clean['_r_offpeak']

degraded = (df_clean['_change'] > 0).sum()
improved = (df_clean['_change'] < 0).sum()
same     = (df_clean['_change'] == 0).sum()
print(f'Деградували поза піком:    {degraded} ({100*degraded/total:.1f}%)')
print(f'Покращились поза піком:    {improved} ({100*improved/total:.1f}%)')
print(f'Без змін:                  {same}     ({100*same/total:.1f}%)')

df_clean.drop(columns=['_r_peak', '_r_offpeak', '_change'], inplace=True)

## 4. DBSCAN

DBSCAN не потребує заздалегідь задавати k — знаходить кластери довільної форми і виявляє викиди.

Ключовий параметр `eps` підбираємо автоматично через **k-distance графік**:
сортуємо відстані до k-го сусіда — "лікоть" кривої вказує оптимальний eps.

In [20]:
from sklearn.neighbors import NearestNeighbors

# K-distance графік на пікових даних для підбору eps
MIN_SAMPLES = cfg['clustering']['dbscan_min_samples']
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_peak_scaled)
distances, _ = nbrs.kneighbors(X_peak_scaled)
k_distances = np.sort(distances[:, -1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances, linewidth=2)
ax.set_xlabel('Точки (відсортовані за відстанню)')
ax.set_ylabel(f'Відстань до {MIN_SAMPLES}-го сусіда')
ax.set_title('K-distance графік для підбору eps')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/05_dbscan_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()

deltas = np.diff(k_distances)
elbow_idx = np.argmax(deltas < deltas[0] * 0.1)
eps_auto = float(round(k_distances[elbow_idx], 3))
print(f'Автоматично підібраний eps: {eps_auto}')

# ── DBSCAN для ПІКОВИХ даних ──────────────────────────────────────────────────
dbscan_peak = DBSCAN(eps=eps_auto, min_samples=MIN_SAMPLES)
df_clean['cluster_dbscan_peak'] = dbscan_peak.fit_predict(X_peak_scaled)

# ── DBSCAN для МІЖПІКОВИХ даних (той самий eps) ───────────────────────────────
dbscan_offpeak = DBSCAN(eps=eps_auto, min_samples=MIN_SAMPLES)
df_clean['cluster_dbscan_offpeak'] = dbscan_offpeak.fit_predict(X_offpeak_scaled)

# Аліас для сумісності
df_clean['cluster_dbscan'] = df_clean['cluster_dbscan_peak']

def safe_metrics(X, labels):
    mask = labels != -1
    if mask.sum() > 1 and len(set(labels[mask])) > 1:
        return (silhouette_score(X[mask], labels[mask]),
                davies_bouldin_score(X[mask], labels[mask]),
                len(set(labels[mask])), int((labels == -1).sum()))
    return 0.0, float('inf'), 0, int((labels == -1).sum())

print(f'\n{"":30s} {"Пік":>10s} {"Міжпік":>10s}')
for metric_name, idx in [('Кластерів', 2), ('Шум (викидів)', 3),
                           ('Silhouette ↑', 0), ('Davies-Bouldin ↓', 1)]:
    vp = safe_metrics(X_peak_scaled,    df_clean['cluster_dbscan_peak'].values)[idx]
    vo = safe_metrics(X_offpeak_scaled, df_clean['cluster_dbscan_offpeak'].values)[idx]
    fmt = '.4f' if isinstance(vp, float) else 'd'
    print(f'{"DBSCAN " + metric_name:30s} {vp:>10{fmt}} {vo:>10{fmt}}')

db_silhouette = safe_metrics(X_peak_scaled, df_clean['cluster_dbscan_peak'].values)[0]
db_db_index   = safe_metrics(X_peak_scaled, df_clean['cluster_dbscan_peak'].values)[1]
n_clusters_db = safe_metrics(X_peak_scaled, df_clean['cluster_dbscan_peak'].values)[2]
n_noise       = safe_metrics(X_peak_scaled, df_clean['cluster_dbscan_peak'].values)[3]

## 5. Порівняння методів

In [21]:
comparison = pd.DataFrame({
    'Метод': ['KMeans Пік', 'KMeans Міжпік', 'DBSCAN Пік', 'DBSCAN Міжпік'],
    'Кластерів': [
        K, K,
        safe_metrics(X_peak_scaled,    df_clean['cluster_dbscan_peak'].values)[2],
        safe_metrics(X_offpeak_scaled, df_clean['cluster_dbscan_offpeak'].values)[2],
    ],
    'Викидів': [
        0, 0,
        safe_metrics(X_peak_scaled,    df_clean['cluster_dbscan_peak'].values)[3],
        safe_metrics(X_offpeak_scaled, df_clean['cluster_dbscan_offpeak'].values)[3],
    ],
    'Silhouette ↑': [
        round(km_peak_silhouette, 4), round(km_offpeak_silhouette, 4),
        round(safe_metrics(X_peak_scaled,    df_clean['cluster_dbscan_peak'].values)[0], 4),
        round(safe_metrics(X_offpeak_scaled, df_clean['cluster_dbscan_offpeak'].values)[0], 4),
    ],
    'Davies-Bouldin ↓': [
        round(km_peak_db, 4), round(km_offpeak_db, 4),
        round(safe_metrics(X_peak_scaled,    df_clean['cluster_dbscan_peak'].values)[1], 4),
        round(safe_metrics(X_offpeak_scaled, df_clean['cluster_dbscan_offpeak'].values)[1], 4),
    ],
})
print('=== ПОРІВНЯННЯ МЕТОДІВ ===')
print(comparison.to_string(index=False))

best_sil = comparison['Silhouette ↑'].max()
best_method = comparison.loc[comparison['Silhouette ↑'].idxmax(), 'Метод']
print(f'\nНайкращий Silhouette Score: {best_sil:.4f} → {best_method}')

## 6. Візуалізація кластерів

In [22]:
CLUSTER_COLORS = {
    'Критична': '#F44336',
    'Погана':   '#FF9800',
    'Добра':    '#8BC34A',
    'Відмінна': '#2196F3',
}
order = ['Критична', 'Погана', 'Добра', 'Відмінна']
palette = [CLUSTER_COLORS[c] for c in order]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 1. Scatter пік ────────────────────────────────────────────────────────────
for label, color in CLUSTER_COLORS.items():
    sub = df_clean[df_clean['label_peak'] == label]
    axes[0, 0].scatter(sub['n_stops'], sub['n_routes'],
                       c=color, label=label, alpha=0.7, s=40,
                       edgecolors='white', linewidth=0.5)
axes[0, 0].set_xlabel('Кількість зупинок')
axes[0, 0].set_ylabel('Кількість маршрутів')
axes[0, 0].set_title('KMeans Пік: зупинки vs маршрути', fontsize=12)
axes[0, 0].legend(title='Кластер')

# ── 2. Scatter міжпік ─────────────────────────────────────────────────────────
for label, color in CLUSTER_COLORS.items():
    sub = df_clean[df_clean['label_offpeak'] == label]
    axes[0, 1].scatter(sub['n_stops'], sub['n_routes'],
                       c=color, label=label, alpha=0.7, s=40,
                       edgecolors='white', linewidth=0.5)
axes[0, 1].set_xlabel('Кількість зупинок')
axes[0, 1].set_ylabel('Кількість маршрутів')
axes[0, 1].set_title('KMeans Міжпік: зупинки vs маршрути', fontsize=12)
axes[0, 1].legend(title='Кластер')

# ── 3. Boxplot score_peak ─────────────────────────────────────────────────────
sns.boxplot(data=df_clean, x='label_peak', y='score_peak',
            order=order, palette=palette, ax=axes[1, 0])
axes[1, 0].set_title('Розподіл score_peak по категоріях', fontsize=12)
axes[1, 0].set_xlabel('Категорія доступності (пік)')
axes[1, 0].set_ylabel('Індекс доступності')

# ── 4. Boxplot score_offpeak ──────────────────────────────────────────────────
sns.boxplot(data=df_clean, x='label_offpeak', y='score_offpeak',
            order=order, palette=palette, ax=axes[1, 1])
axes[1, 1].set_title('Розподіл score_offpeak по категоріях', fontsize=12)
axes[1, 1].set_xlabel('Категорія доступності (міжпік)')
axes[1, 1].set_ylabel('Індекс доступності')

plt.suptitle('KMeans кластеризація: пік vs міжпік', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/08_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [23]:
pca = PCA(n_components=2)
X_pca_peak = pca.fit_transform(X_peak_scaled)

pca2 = PCA(n_components=2)
X_pca_offpeak = pca2.fit_transform(X_offpeak_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for label, color in CLUSTER_COLORS.items():
    mask = df_clean['label_peak'] == label
    ax1.scatter(X_pca_peak[mask, 0], X_pca_peak[mask, 1],
                c=color, label=label, alpha=0.75, s=45,
                edgecolors='white', linewidth=0.5)
ax1.set_title(f'KMeans Пік — PCA ({pca.explained_variance_ratio_.sum():.1%})', fontsize=12)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax1.legend(title='Категорія')

for label, color in CLUSTER_COLORS.items():
    mask = df_clean['label_offpeak'] == label
    ax2.scatter(X_pca_offpeak[mask, 0], X_pca_offpeak[mask, 1],
                c=color, label=label, alpha=0.75, s=45,
                edgecolors='white', linewidth=0.5)
ax2.set_title(f'KMeans Міжпік — PCA ({pca2.explained_variance_ratio_.sum():.1%})', fontsize=12)
ax2.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax2.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax2.legend(title='Категорія')

plt.tight_layout()
plt.savefig('../outputs/09_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [24]:
cols = [
    'facility_id', 'facility_type', 'name', 'lon', 'lat',
    'n_stops', 'n_routes',
    'frequency_peak', 'frequency_offpeak',
    'score_peak', 'score_offpeak', 'score_delta',
    'accessibility_score',        # = score_peak, для сумісності
    'cluster_peak',    'label_peak',
    'cluster_offpeak', 'label_offpeak',
    'cluster_kmeans',  'accessibility_label',   # аліаси для NB07
    'cluster_dbscan',  'cluster_dbscan_peak', 'cluster_dbscan_offpeak',
]
df_clean[cols].to_csv('../data/processed/clustered_facilities.csv', index=False, encoding='utf-8')

print('Файл збережено: data/processed/clustered_facilities.csv')
print()
print('=== ФІНАЛЬНИЙ РОЗПОДІЛ (ПІК) ===')
for label in ['Відмінна', 'Добра', 'Погана', 'Критична']:
    count = len(df_clean[df_clean['label_peak'] == label])
    pct   = 100 * count / len(df_clean)
    h = len(df_clean[(df_clean['label_peak']==label) & (df_clean['facility_type']=='hospital')])
    s = len(df_clean[(df_clean['label_peak']==label) & (df_clean['facility_type']=='school')])
    print(f'  {label:12s}: {count:4d} ({pct:.1f}%) | лікарні: {h}, школи: {s}')

print()
print('=== ФІНАЛЬНИЙ РОЗПОДІЛ (МІЖПІК) ===')
for label in ['Відмінна', 'Добра', 'Погана', 'Критична']:
    count = len(df_clean[df_clean['label_offpeak'] == label])
    pct   = 100 * count / len(df_clean)
    print(f'  {label:12s}: {count:4d} ({pct:.1f}%)')

print()
print('Notebook 05 виконано. Переходьте до 06_regression.ipynb')